<a href="https://colab.research.google.com/github/aadithya-vimal/SamsungInnovationCampus/blob/main/practice_central_limit_theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Notebook: Central Limit Theorem (CLT)

This notebook demonstrates the **Central Limit Theorem** using Python in Google Colab.

We will use:
- **NumPy** for simulation
- **Matplotlib** for visualization
- **SciPy** for normal distribution comparison

---

## Learning Goals

By the end of this notebook, you should be able to:

1. explain the idea behind the Central Limit Theorem,
2. simulate repeated sampling from different populations,
3. observe how the **sampling distribution of the sample mean** becomes approximately normal,
4. understand the role of **sample size**,
5. relate CLT to real-world data science and AI problems.


## 1. Why Do We Need the Central Limit Theorem?

In statistics and machine learning, we often work with **sample averages** rather than the full population.

Examples:
- average test score of students,
- average response time of a server,
- average loss over a batch in model training,
- average number of clicks per user.

The Central Limit Theorem explains why averages are so useful.

---

## 2. Statement of the Central Limit Theorem

If:

- a population has mean \(\mu\),
- population standard deviation \(\sigma\),
- we repeatedly draw samples of size \(n\),
- and compute the sample mean \(\bar{X}\),

then, for sufficiently large \(n\), the distribution of \(\bar{X}\) becomes approximately **normal**, even if the original population is not normal.

### Key Results

The sample mean has:

\[
E(\bar{X}) = \mu
\]

\[
Var(\bar{X}) = \frac{\sigma^2}{n}
\]

\[
SD(\bar{X}) = \frac{\sigma}{\sqrt{n}}
\]

The quantity

\[
\frac{\bar{X} - \mu}{\sigma / \sqrt{n}}
\]

approximately follows a standard normal distribution for large \(n\).

---

## Important Intuition

The CLT is **not** about the original data becoming normal.

It is about the **distribution of sample means** becoming normal.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)

## 3. A Function to Simulate Sample Means

This helper function:

1. draws many samples from a population,
2. computes the mean of each sample,
3. returns the collection of sample means.


In [ ]:
def simulate_sample_means(generator_function, sample_size=5, num_samples=5000):
    means = []
    for _ in range(num_samples):
        sample = generator_function(sample_size)
        means.append(np.mean(sample))
    return np.array(means)

## 4. Case 1: Uniform Population

A uniform population is flat. It is not bell-shaped like a normal distribution.

We will:
- generate data from a uniform distribution,
- take repeated samples,
- compute sample means,
- compare results for different sample sizes.


In [ ]:
uniform_population = np.random.uniform(0, 1, 100000)

plt.figure(figsize=(8,4))
plt.hist(uniform_population, bins=30, edgecolor='black')
plt.title('Original Population: Uniform(0,1)')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

print("Population mean:", uniform_population.mean())
print("Population std:", uniform_population.std())

In [ ]:
sample_sizes = [2, 5, 10, 30]

plt.figure(figsize=(12, 8))

for i, n in enumerate(sample_sizes, 1):
    means = simulate_sample_means(lambda size: np.random.uniform(0, 1, size), sample_size=n, num_samples=5000)
    plt.subplot(2, 2, i)
    plt.hist(means, bins=30, density=True, edgecolor='black')
    plt.title(f'Sampling Distribution of Mean (n={n})')
    plt.xlabel('Sample Mean')
    plt.ylabel('Density')

plt.tight_layout()
plt.show()

### Observation

As sample size increases:
- the distribution of the sample mean becomes more bell-shaped,
- the spread becomes smaller,
- the means concentrate near the true population mean.


## 5. Overlay the Theoretical Normal Curve

For a uniform distribution on \([0,1]\):

\[
\mu = 0.5
\]

\[
\sigma = \sqrt{\frac{1}{12}}
\]

So the sampling distribution of the mean should be approximately normal with:

\[
\mu_{\bar{X}} = 0.5
\]

\[
\sigma_{\bar{X}} = \frac{\sqrt{1/12}}{\sqrt{n}}
\]


In [ ]:
n = 30
means = simulate_sample_means(lambda size: np.random.uniform(0, 1, size), sample_size=n, num_samples=5000)

mu = 0.5
sigma = np.sqrt(1/12)
se = sigma / np.sqrt(n)

x = np.linspace(means.min(), means.max(), 300)

plt.figure(figsize=(8,4))
plt.hist(means, bins=30, density=True, alpha=0.7, edgecolor='black', label='Simulated sample means')
plt.plot(x, norm.pdf(x, loc=mu, scale=se), linewidth=2, label='Theoretical normal curve')
plt.title(f'CLT for Uniform Population (n={n})')
plt.xlabel('Sample Mean')
plt.ylabel('Density')
plt.legend()
plt.show()

## 6. Case 2: Exponential Population

An exponential distribution is strongly right-skewed.  
This makes it a very good example for the CLT, because the original population is clearly **not normal**.

Let us see what happens to the sampling distribution of the mean.


In [ ]:
exp_population = np.random.exponential(scale=1.0, size=100000)

plt.figure(figsize=(8,4))
plt.hist(exp_population, bins=40, edgecolor='black')
plt.title('Original Population: Exponential(scale=1)')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

print("Population mean:", exp_population.mean())
print("Population std:", exp_population.std())

In [ ]:
sample_sizes = [2, 5, 10, 30]

plt.figure(figsize=(12, 8))

for i, n in enumerate(sample_sizes, 1):
    means = simulate_sample_means(lambda size: np.random.exponential(scale=1.0, size=size), sample_size=n, num_samples=5000)
    plt.subplot(2, 2, i)
    plt.hist(means, bins=30, density=True, edgecolor='black')
    plt.title(f'Exponential Population → Sample Mean (n={n})')
    plt.xlabel('Sample Mean')
    plt.ylabel('Density')

plt.tight_layout()
plt.show()

### Observation

Even though the exponential population is skewed:

- for small \(n\), the sample-mean distribution is also skewed,
- as \(n\) increases, it becomes more symmetric and normal-like.

This is the Central Limit Theorem in action.


## 7. Mean and Standard Error of the Sample Mean

The CLT tells us that:

\[
E(\bar{X}) = \mu
\]

and

\[
SD(\bar{X}) = \frac{\sigma}{\sqrt{n}}
\]

Let us verify this numerically.


In [ ]:
population_mean = 1.0
population_std = 1.0

for n in [5, 10, 30, 50]:
    means = simulate_sample_means(lambda size: np.random.exponential(scale=1.0, size=size), sample_size=n, num_samples=10000)
    empirical_mean = means.mean()
    empirical_std = means.std()
    theoretical_se = population_std / np.sqrt(n)

    print(f"n = {n}")
    print(f" Empirical mean of sample means  = {empirical_mean:.4f}")
    print(f" Theoretical population mean     = {population_mean:.4f}")
    print(f" Empirical std of sample means   = {empirical_std:.4f}")
    print(f" Theoretical standard error      = {theoretical_se:.4f}")
    print("-" * 50)

## 8. Case 3: Discrete Population (Binomial)

The CLT also works for discrete distributions.

Let us use a binomial population:
- number of successes in 10 trials,
- success probability = 0.3


In [ ]:
binom_population = np.random.binomial(n=10, p=0.3, size=100000)

plt.figure(figsize=(8,4))
plt.hist(binom_population, bins=np.arange(-0.5, 11.5, 1), edgecolor='black', density=True)
plt.title('Original Population: Binomial(n=10, p=0.3)')
plt.xlabel('Value')
plt.ylabel('Density')
plt.show()

print("Population mean:", binom_population.mean())
print("Population std:", binom_population.std())

In [ ]:
sample_sizes = [2, 5, 10, 30]

plt.figure(figsize=(12, 8))

for i, n in enumerate(sample_sizes, 1):
    means = simulate_sample_means(lambda size: np.random.binomial(n=10, p=0.3, size=size), sample_size=n, num_samples=5000)
    plt.subplot(2, 2, i)
    plt.hist(means, bins=30, density=True, edgecolor='black')
    plt.title(f'Binomial Population → Sample Mean (n={n})')
    plt.xlabel('Sample Mean')
    plt.ylabel('Density')

plt.tight_layout()
plt.show()

## 9. Standardization and Z-score Form

The CLT often appears in the standardized form:

\[
Z = \frac{\bar{X} - \mu}{\sigma / \sqrt{n}}
\]

For sufficiently large \(n\), this is approximately standard normal.

Let us verify this for exponential data with \(n = 30\).


In [ ]:
n = 30
mu = 1.0
sigma = 1.0

means = simulate_sample_means(lambda size: np.random.exponential(scale=1.0, size=size), sample_size=n, num_samples=10000)
z_scores = (means - mu) / (sigma / np.sqrt(n))

x = np.linspace(-4, 4, 300)

plt.figure(figsize=(8,4))
plt.hist(z_scores, bins=35, density=True, alpha=0.7, edgecolor='black', label='Simulated Z values')
plt.plot(x, norm.pdf(x), linewidth=2, label='Standard normal')
plt.title('Standardized Sample Means')
plt.xlabel('Z')
plt.ylabel('Density')
plt.legend()
plt.show()

## 10. Real-World Example 1: Average Daily App Usage

Suppose daily app usage time per user is highly skewed:
- many users use the app for a short time,
- a few users use it for a very long time.

This can be modeled by an exponential-like distribution.

Now suppose we repeatedly sample groups of users and compute the **average usage time**.  
The CLT tells us the distribution of those averages becomes approximately normal.


In [ ]:
n = 40
means = simulate_sample_means(lambda size: np.random.exponential(scale=2.0, size=size), sample_size=n, num_samples=5000)

plt.figure(figsize=(8,4))
plt.hist(means, bins=30, edgecolor='black', density=True)
plt.title('Sampling Distribution of Average App Usage Time')
plt.xlabel('Average Usage Time')
plt.ylabel('Density')
plt.show()

print("Estimated mean of sample means:", means.mean())

## 11. Real-World Example 2: Batch Loss in Machine Learning

During neural network training, the loss value for individual samples may vary a lot.

But when we compute the **average loss over a mini-batch**, that average is more stable.

This is one intuitive way the CLT connects to machine learning:
- individual samples may be noisy,
- averages over batches are better behaved.


In [ ]:
loss_means = simulate_sample_means(lambda size: np.random.gamma(shape=2.0, scale=1.5, size=size),
                                   sample_size=32, num_samples=5000)

plt.figure(figsize=(8,4))
plt.hist(loss_means, bins=30, edgecolor='black', density=True)
plt.title('Sampling Distribution of Average Mini-batch Loss')
plt.xlabel('Average Batch Loss')
plt.ylabel('Density')
plt.show()

## 12. Effect of Sample Size on Spread

The spread of the sampling distribution is measured by the **standard error**:

\[
SE = \frac{\sigma}{\sqrt{n}}
\]

This means:
- increasing sample size reduces uncertainty,
- larger samples produce more stable averages.


In [ ]:
population_std = 1.0
n_values = np.array([2, 5, 10, 20, 30, 50, 100])
se_values = population_std / np.sqrt(n_values)

plt.figure(figsize=(8,4))
plt.plot(n_values, se_values, marker='o')
plt.title('Standard Error Decreases as Sample Size Increases')
plt.xlabel('Sample Size n')
plt.ylabel('Standard Error')
plt.show()

## 13. Common Misunderstandings

### Misconception 1
**“The CLT says the original population becomes normal.”**  
No. It says the **sampling distribution of the mean** becomes normal.

### Misconception 2
**“The CLT works only for normal populations.”**  
No. It is especially useful because it works even when the population is not normal.

### Misconception 3
**“A sample size of 30 always guarantees normality.”**  
Not always. It depends on how skewed or unusual the original population is.  
But \(n=30\) is often a practical rule of thumb.


## 14. Practice Exercises

Try these on your own.

### Exercise 1
Generate a population from:
- Uniform(10, 20)

Take repeated samples and plot the sampling distribution of the sample mean for:
- \(n=2\)
- \(n=10\)
- \(n=50\)

### Exercise 2
Use an exponential population and verify numerically that:

- the mean of sample means is close to the population mean,
- the standard deviation of sample means is close to \(\sigma/\sqrt{n}\)

for:
- \(n=5\)
- \(n=20\)
- \(n=100\)

### Exercise 3
Create a right-skewed custom population using:
`np.random.gamma(shape=2, scale=3, size=100000)`

Then show how the sample-mean distribution changes with sample size.

### Exercise 4
Standardize the sample means using the Z-score formula and compare the histogram with the standard normal curve.

### Exercise 5
Suppose response times of a website are skewed.  
Model this using an exponential or gamma distribution.  
Use the CLT to study the average response time for groups of 50 requests.


In [ ]:
# Write your solutions here


## 15. Mini Challenge

Choose one real-world problem and model it using the CLT.

Possible ideas:
- average marks of students,
- average waiting time in a hospital,
- average number of clicks on an ad,
- average latency in a network,
- average batch loss in training.

### Your Task
1. choose a population distribution,
2. justify why it makes sense,
3. simulate repeated samples,
4. plot the sampling distribution of the mean,
5. explain how the CLT helps in understanding that problem.


## Final Takeaway

The Central Limit Theorem is one of the most important ideas in statistics because it explains why averages are so powerful.

It helps us:
- estimate unknown population values,
- build confidence intervals,
- perform hypothesis tests,
- understand uncertainty in data science and AI.

Even when raw data is messy, skewed, or discrete, the distribution of sample means often becomes nicely structured and approximately normal.
